# full_data 2017 与汇算数据匹配检查（交互版）

这个 notebook 按步骤检查 2017 年 `full_data.dta` 企业和汇算数据的匹配情况。

特点：

- 不写函数；
- 每一步单独运行；
- 每一步都能看到样本量和匹配率；
- 匹配时只保留 `full_data` 主样本企业，不引入 using-only 企业。


## 0. 设置路径和基础参数


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path(r'G:\Kuangyu_Temp\Outsource')
PROJECT = ROOT / 'productivity'
PROJECT_INNER = PROJECT / 'productivity'

FULL_DATA = ROOT / 'full_data.dta'
BRIDGE = PROJECT / 'cid_entid_unique.dta'
HUISUAN_2017 = Path(r'H:\汇算数据\2017.dta')
OUT_CSV = PROJECT_INNER / 'full_data_2017_huisuan_match_check_python.csv'
OUT_DTA = PROJECT_INNER / 'full_data_2017_huisuan_match_check_python.dta'

CHUNKSIZE = 500_000

print('FULL_DATA:', FULL_DATA, '| exists =', FULL_DATA.exists())
print('BRIDGE:', BRIDGE, '| exists =', BRIDGE.exists())
print('HUISUAN_2017:', HUISUAN_2017, '| exists =', HUISUAN_2017.exists())


## 1. 从 full_data.dta 中提取 2017 年企业

这一步只读取 `firm_id` 和 `year` 两列，并且用分块方式读取，避免一次性把整个 `full_data.dta` 放进内存。


In [ ]:
firm_ids = set()
rows_seen = 0
rows_2017_seen = 0

reader = pd.read_stata(
    FULL_DATA,
    columns=['firm_id', 'year'],
    chunksize=CHUNKSIZE,
    convert_categoricals=False,
)

for i, chunk in enumerate(reader, start=1):
    rows_seen = rows_seen + len(chunk)
    chunk_2017 = chunk.loc[chunk['year'] == 2017, ['firm_id']].copy()
    rows_2017_seen = rows_2017_seen + len(chunk_2017)

    chunk_2017['firm_id'] = (
        chunk_2017['firm_id']
        .astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )

    firm_ids.update(chunk_2017['firm_id'].dropna().unique().tolist())

    print(
        f'chunk {i:>3}: total rows seen = {rows_seen:,}, '
        f'2017 rows seen = {rows_2017_seen:,}, '
        f'unique firm_id so far = {len(firm_ids):,}'
    )


## 2. 生成 2017 年唯一企业表

这一步对应 Stata 里的 `keep if year == 2017` 和 `bysort firm_id: keep if _n == 1`。


In [ ]:
full2017 = pd.DataFrame({'firm_id': sorted(firm_ids)})
full2017['cid'] = pd.to_numeric(full2017['firm_id'], errors='coerce')

n_full = len(full2017)
n_missing_cid = int(full2017['cid'].isna().sum())

print('full_data 2017 unique firms:', f'{n_full:,}')
print('missing cid after converting firm_id to numeric:', f'{n_missing_cid:,}')
print('missing cid rate:', f'{n_missing_cid / n_full:.2%}')

full2017.head()


## 3. 读取桥接表 cid_entid_unique.dta

这一步先查看桥接表本身有多少行，以及 `cid` 是否重复。


In [ ]:
bridge = pd.read_stata(
    BRIDGE,
    columns=['cid', 'id'],
    convert_categoricals=False,
)

print('bridge raw rows:', f'{len(bridge):,}')
print('bridge duplicated cid rows:', f"{bridge.duplicated(subset=['cid']).sum():,}")
print('bridge missing cid:', f"{bridge['cid'].isna().sum():,}")
print('bridge missing id:', f"{bridge['id'].isna().sum():,}")

bridge.head()


## 4. 每个 cid 保留第一行桥接记录

这一步是为了保证后面可以做多对一匹配。


In [ ]:
bridge_unique = bridge.dropna(subset=['cid']).drop_duplicates(subset=['cid'], keep='first').copy()

print('bridge rows after keeping first row per cid:', f'{len(bridge_unique):,}')
print('unique cid in bridge_unique:', f"{bridge_unique['cid'].nunique():,}")

bridge_unique.head()


## 5. 把 full_data 2017 企业匹配到桥接表

这里用左连接，等价于 Stata 里的 `merge ..., keep(master match)`。

因此，这一步不会把桥接表里额外的 using-only 企业混进来。


In [ ]:
full_bridge = full2017.merge(
    bridge_unique,
    on='cid',
    how='left',
    indicator='bridge_merge',
)

full_bridge['bridge_matched'] = full_bridge['bridge_merge'] == 'both'
n_bridge = int(full_bridge['bridge_matched'].sum())

print('full_data 2017 unique firms:', f'{n_full:,}')
print('matched to cid_entid_unique.dta:', f'{n_bridge:,}')
print('bridge match rate:', f'{n_bridge / n_full:.2%}')

full_bridge['bridge_merge'].value_counts(dropna=False)


In [ ]:
full_bridge.head()


## 6. 读取 2017 年汇算数据

这里只读取 `id`、`从业人数`、`资产总额` 三列。


In [ ]:
huisuan = pd.read_stata(
    HUISUAN_2017,
    columns=['id', '从业人数', '资产总额'],
    convert_categoricals=False,
)

print('huisuan raw rows:', f'{len(huisuan):,}')
print('huisuan duplicated id rows:', f"{huisuan.duplicated(subset=['id']).sum():,}")
print('huisuan missing id:', f"{huisuan['id'].isna().sum():,}")
print('huisuan missing employment:', f"{huisuan['从业人数'].isna().sum():,}")
print('huisuan missing assets:', f"{huisuan['资产总额'].isna().sum():,}")

huisuan.head()


## 7. 每个汇算 id 保留第一行

这一步对应 Stata 里的 `bysort id year: gen rep_no = _n` 和 `keep if rep_no == 1`。

因为当前文件本身就是 2017 年汇算数据，所以直接按 `id` 去重即可。


In [ ]:
huisuan_first = huisuan.dropna(subset=['id']).drop_duplicates(subset=['id'], keep='first').copy()

print('huisuan rows after keeping first row per id:', f'{len(huisuan_first):,}')
print('unique id in huisuan_first:', f"{huisuan_first['id'].nunique():,}")

huisuan_first.head()


## 8. 用 id 匹配汇算数据

这里也用左连接，等价于 Stata 里的 `merge ..., keep(master match)`。

因此最终分母仍然是 `full_data.dta` 中 2017 年唯一企业数。


In [ ]:
full_match = full_bridge.merge(
    huisuan_first,
    on='id',
    how='left',
    indicator='huisuan_merge',
)

full_match['huisuan_matched'] = full_match['huisuan_merge'] == 'both'

n_huisuan = int(full_match['huisuan_matched'].sum())
n_emp = int(full_match['从业人数'].notna().sum())
n_asset = int(full_match['资产总额'].notna().sum())

print('full_data 2017 unique firms:', f'{n_full:,}')
print('matched to huisuan 2017:', f'{n_huisuan:,}')
print('huisuan match rate:', f'{n_huisuan / n_full:.2%}')
print('nonmissing employment:', f'{n_emp:,}', '| rate =', f'{n_emp / n_full:.2%}')
print('nonmissing assets:', f'{n_asset:,}', '| rate =', f'{n_asset / n_full:.2%}')

full_match['huisuan_merge'].value_counts(dropna=False)


In [ ]:
full_match.head()


## 9. 汇总匹配情况


In [ ]:
summary = pd.DataFrame([
    ['full_data 2017 unique firms', n_full, n_full, n_full / n_full],
    ['matched to cid_entid_unique.dta', n_bridge, n_full, n_bridge / n_full],
    ['matched to huisuan 2017', n_huisuan, n_full, n_huisuan / n_full],
    ['nonmissing employment', n_emp, n_full, n_emp / n_full],
    ['nonmissing assets', n_asset, n_full, n_asset / n_full],
], columns=['step', 'count', 'denominator', 'rate'])

summary['rate_percent'] = summary['rate'].map(lambda x: f'{x:.2%}')
summary


## 10. 交叉检查：桥接成功和汇算成功的关系


In [ ]:
pd.crosstab(
    full_match['bridge_matched'],
    full_match['huisuan_matched'],
    rownames=['bridge_matched'],
    colnames=['huisuan_matched'],
    margins=True,
)


## 11. 可选：保存检查结果

如果只是想看每一步匹配情况，可以不运行这一步。

如果要保存完整检查数据，可以运行下面这个 cell。


In [ ]:
cols_to_save = [
    'firm_id', 'cid', 'id',
    'bridge_matched', 'huisuan_matched',
    '从业人数', '资产总额',
]

full_match[cols_to_save].to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print('saved CSV:', OUT_CSV)

# 如果需要 Stata dta 文件，再取消下面两行注释。
# full_match[cols_to_save].to_stata(OUT_DTA, write_index=False, version=118)
# print('saved DTA:', OUT_DTA)
